# 01 — Electric Field (Poisson) Solver  


---
## 1. Imports

In [ ]:
import sys, os, warnings
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# Add project root to path so we can import src.*
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.core.grid import Grid1D
from src.core.device import Device
from src.core.layer import Layer
from src.poisson.solver import PoissonSolver
from src.core.constants import q, VT
from src.utils.ingestion import DataIngestionService, DataIngestionConfig
from src.simulator import SPADSimulator
%matplotlib inline
print('Imports OK')

---
## 2. Setting up the grid

The `Grid1D` class creates a uniform 1D mesh over $[0, L]$ with $N$ nodes. The spacing is $\Delta x = L / (N-1)$.

In [ ]:
L = 5.0e-4  # 5 µm total device thickness
N = 201      # grid nodes
grid = Grid1D(L=L, N=N)

print(f"dx = {grid.dx:.3e} cm")
print(f"x  = {grid.x[:5]}")
print(f"Number of nodes: {grid.no_of_nodes}")

### The gradient helper

The grid also provides a `gradient` method that computes $E(x) = -d\phi/dx$ via `np.gradient` (central differences in the interior, one-sided at boundaries):

In [ ]:
# Quick test: phi(x) = x^2 => E(x) = -2x
phi_test = grid.x ** 2
E_test = grid.gradient(phi_test)

plt.figure(figsize=(10, 3))
plt.plot(grid.x * 1e4, E_test, 'b-', label='E(x) from gradient')
plt.plot(grid.x * 1e4, -2 * grid.x, 'r--', label='-2x (exact)')
plt.xlabel('Depth (µm)')
plt.ylabel('E(x)')
plt.legend()
plt.grid(alpha=0.3)
plt.title(r'Gradient test: $\phi(x) = x^2$')
plt.show()

---
## 3. The Poisson equation

### 3.1 Governing equation

The electrostatic potential $\phi(x)$ satisfies the 1D Poisson equation:

$$
\frac{d}{dx}\left(\varepsilon(x) \frac{d\phi}{dx}\right) = -\rho(x)
$$

where $\rho(x)$ is the **total charge density** including mobile carriers:

$$
\rho(x) = q \bigl(p(x) - n(x) + N_D^+(x) - N_A^-(x)\bigr)
$$

### 3.2 Boltzmann statistics

For non-degenerate semiconductors, electron and hole densities follow Boltzmann statistics:

$$
n(x) = n_i(x) \exp\!\left(\frac{\phi(x) - \phi_n}{V_T}\right) \\
p(x) = n_i(x) \exp\!\left(\frac{\phi_p - \phi(x)}{V_T}\right)
$$

where $\phi_n$ and $\phi_p$ are the quasi-Fermi levels (for equilibrium we set $\phi_n = \phi_p = V_{\text{bias}}$), $n_i$ is the intrinsic carrier density, and $V_T = k_B T / q$ is the thermal voltage.

### 3.3 The nonlinear residual

Substituting the carrier densities into $\rho$, the Poisson equation becomes **nonlinear** in $\phi$ because $n$ and $p$ depend exponentially on $\phi$. We need a Newton–Raphson iteration.

---
## 4. Finite-volume discretization

### 4.1 The flux form

At each interior node $i$ (away from boundaries), we discretize:

$$
\frac{\varepsilon_{i+1/2}(\phi_{i+1} - \phi_i) - \varepsilon_{i-1/2}(\phi_i - \phi_{i-1})}{\Delta x^2} = -\rho_i
$$

where $\varepsilon_{i+1/2} = (\varepsilon_i + \varepsilon_{i+1})/2$ is the permittivity at the **half-grid** (interface between nodes), computed in the solver as:

```python
self._eps_half = 0.5 * (self._eps[:-1] + self._eps[1:])
```

This gives the residual vector $F(\phi)$:

$$
F_i(\phi) = \frac{\varepsilon_{i+1/2}(\phi_{i+1} - \phi_i) - \varepsilon_{i-1/2}(\phi_i - \phi_{i-1})}{\Delta x^2} + \rho_i(\phi_i)
$$

### 4.2 Jacobian

Newton's method requires $\partial F_i / \partial \phi_j$. The discretization gives a **tridiagonal** Jacobian:

$$
J_{i,i-1} = \frac{\varepsilon_{i-1/2}}{\Delta x^2} \\
J_{i,i}   = -\frac{\varepsilon_{i+1/2} + \varepsilon_{i-1/2}}{\Delta x^2} + \frac{\partial \rho_i}{\partial \phi_i} \\
J_{i,i+1} = \frac{\varepsilon_{i+1/2}}{\Delta x^2}
$$

where the carrier derivative is:

$$
\frac{\partial \rho}{\partial \phi} = -\frac{q}{V_T}(n + p)
$$

---
## 5. Boundary conditions

Dirichlet conditions at both contacts:

**Left contact (p-side, $x=0$):**
$$
\phi(0) = -V_T \ln\!\left(\frac{N_A}{n_i}\right) \quad \text{(built-in potential from p-type doping)}
$$

**Right contact (n-side, $x=L$):**
$$
\phi(L) = V_{\text{bias}} + V_T \ln\!\left(\frac{N_D}{n_i}\right) \quad \text{(applied bias + n-type contact)}
$$

The solver computes these in `_contact_bias()`:

In [ ]:
from src.core.material import Material
from src.core.constants import VT

# Demonstrate contact bias calculation
T = 300.0
vth = VT(T)
print(f"Thermal voltage V_T({T}K) = {vth:.4f} V")

# Example: p-side doping 1e18 cm^-3, ni ~ 1e10 cm^-3
na_left = 1e18
ni = 1e10
phi_0 = -vth * np.log(na_left / ni)
print(f"Example left contact phi_0 = {phi_0:.3f} V  (p-side, N_A={na_left:.1e})")

# Example: n-side doping 1e17 cm^-3, Vbias = 0 V
nd_right = 1e17
phi_L = 0.0 + vth * np.log(nd_right / ni)
print(f"Example right contact phi_L = {phi_L:.3f} V  (n-side, Vbias=0, N_D={nd_right:.1e})")

---
## 6. Newton-Raphson loop

At each iteration $k$:

1. Compute residual $F(\phi^k)$
2. Compute Jacobian $J(\phi^k)$
3. Solve linear system: $J \Delta\phi = -F$ (using banded/LAPACK solver)
4. Update: $\phi^{k+1} = \phi^k + \alpha \Delta\phi$ (with damping $\alpha$)
5. Check convergence: $\|F\|_2 < \text{tol}$

The hot loop is accelerated with Numba (`newton_iteration` in `numba_solver.py`), but the structure is the same.

### Initial guess

A good initial guess matters. The solver uses a **tanh step** centered at the metallurgical junction:

$$
\phi^{(0)}(x) = \phi_0 + (\phi_L - \phi_0) \cdot \frac{1}{2}\left[1 + \tanh\!\left(\frac{x - x_j}{W}\right)\right]
$$

where $x_j$ is the junction position (where net doping changes sign) and $W$ is a smoothing width. Below built-in, a **depletion approximation** provides the guess.

In [ ]:
# Demonstrate the step initial guess
x = grid.x
phi_0 = -0.8
phi_L = 0.8
x_j = 1.0e-4  # 1 µm junction
W = 0.1e-4     # 0.1 µm smoothing

phi_guess = phi_0 + (phi_L - phi_0) * 0.5 * (1 + np.tanh((x - x_j) / W))

plt.figure(figsize=(8, 3))
plt.plot(x * 1e4, phi_guess)
plt.axvline(x_j * 1e4, color='r', ls='--', alpha=0.4, label='junction')
plt.xlabel('Depth (µm)')
plt.ylabel('$\\phi$ (V)')
plt.grid(alpha=0.3)
plt.title('Initial guess: tanh step')
plt.legend()
plt.show()

---
## 7. Running the full solver on the SPAD device

Now let's build the actual SAGCM SPAD device and solve Poisson at several bias voltages.

In [ ]:
# Build device and simulator
config = DataIngestionConfig.from_defaults()
svc = DataIngestionService(config)
sim = svc.build_simulator()
device = sim.device

print(f"Device: {len(device.layers)} layers")
print(f"Total thickness: {device.L*1e4:.2f} µm")
print(f"Grid nodes: {device.no_of_nodes}")
print(f"Breakdown voltage: {sim.Vbr:.1f} V" if hasattr(sim, 'Vbr') else "")

In [ ]:
# Create the Poisson solver
mat = device.material
eps_grid = np.array([device.materials[name].eps_r for name in mat.mat_name])
ni_grid = np.array([device.materials[name].ni() for name in mat.mat_name])

solver = PoissonSolver(
    grid=device.grid,
    T=device.T,
    doping=device.doping,
    eps_grid=eps_grid,
    ni_grid=ni_grid,
    tol=1e-3,
    max_iter=300,
    damp=0.5
)

print(f"PoissonSolver created")
print(f"  grid: N={solver.grid.no_of_nodes}, L={solver.grid.L:.3e} cm")
print(f"  T={solver.T} K")

In [ ]:
# Solve at several bias voltages using the simulator (robust incremental stepping)
# The simulator's solve_poisson caches and reuses the previous solution as a guess.

biases = [30.0, 40.0, 50.0, 60.0]
solutions = {}

Vbr, _ = sim.find_breakdown(V_start=20, V_max=80, V_step=1.0)
print(f"Breakdown voltage: Vbr = {Vbr:.1f} V\n")

for V in biases:
    phi, E, info = sim.solve_poisson(float(V))
    solutions[V] = (phi, E)
    print(f"V = {V:5.1f} V  |  phi_max = {phi.max():7.2f} V  |  |E|_max = {np.max(np.abs(E)):8.2e} V/cm  |  iters = {info['iterations']}")

---
## 8. Plotting electric field and potential

In [ ]:
# Electrostatic potential phi(x)
plt.figure(figsize=(10, 4))
for V in biases:
    phi, E = solutions[V]
    plt.plot(device.grid.x * 1e4, phi, lw=2, label=f'V = {V} V')
plt.xlabel('Depth (µm)')
plt.ylabel('$\\phi$ (V)')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.title('Electrostatic Potential vs Depth')
plt.show()

In [ ]:
# Electric field E(x) = -dphi/dx
plt.figure(figsize=(10, 4))
for V in biases:
    phi, E = solutions[V]
    plt.plot(device.grid.x * 1e4, np.abs(E), lw=2, label=f'V = {V} V')

# Mark the charge sheet region
plt.axvspan(0.85, 1.2, color='yellow', alpha=0.15, label='Charge sheet (n+ InP)')
plt.xlabel('Depth (µm)')
plt.ylabel('$|E(x)|$ (V/cm)')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.title('Electric Field Profile')
plt.yscale('log')
plt.show()

**Key observations:**

1. The charge sheet (n+ InP) creates a **sharp field drop** — the multiplication region (left of it) sees high field ($> 4\times10^5$ V/cm), while the absorber (right of it) stays at low field ($< 10^5$ V/cm).
2. As bias increases, the field in the multiplication region rises uniformly — the peak shifts but roughly stays in the same location.
3. The potential $\phi(x)$ is almost linear in the high-field region (constant field), which is the hallmark of a **fully depleted** multiplication layer.

---
## 9. Understanding the carrier densities

Let's look at $n(x)$ and $p(x)$ from the solution — this shows the depletion region:

In [ ]:
def plot_carriers(phi, Vbias):
    vth = VT(device.T)
    x = device.grid.x
    ni = solver._ni
    phi_n = Vbias
    phi_p = 0.0
    
    # Use the solver's carrier density method
    n, p = solver._carrier_densities(phi, phi_n, phi_p)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
    
    ax1.plot(x * 1e4, n, 'b-', label='n(x)')
    ax1.plot(x * 1e4, p, 'r-', label='p(x)')
    ax1.set_yscale('log')
    ax1.set_xlabel('Depth (µm)')
    ax1.set_ylabel('Carrier density (cm⁻³)')
    ax1.legend()
    ax1.grid(alpha=0.3)
    ax1.set_title(f'Carrier densities at V = {Vbias} V')
    
    # Net charge
    net = device.net_doping_on_grid
    rho = solver._q_val * (p - n + net)
    ax2.plot(x * 1e4, rho, 'k-', lw=1.5)
    ax2.axhline(0, color='gray', ls='--')
    ax2.set_xlabel('Depth (µm)')
    ax2.set_ylabel('$\\rho$ (C/cm³)')
    ax2.grid(alpha=0.3)
    ax2.set_title('Net charge density')
    
    plt.tight_layout()
    plt.show()

phi_30, E_30 = solutions[30.0]
plot_carriers(phi_30, 30.0)
phi_60, E_60 = solutions[60.0]
plot_carriers(phi_60, 60.0)

---## 11. Interactive explorationDrag the bias slider to see how $E(x)$ and $\phi(x)$ change in real time.

In [ ]:
from ipywidgets import FloatSlider, interactive, VBox, Outputfrom IPython.display import displayE_slider = FloatSlider(value=60, min=20, max=80, step=0.5,                       description='Bias (V):', continuous_update=False,                       style={'description_width': '80px'},                       layout={'width': '400px'})out = Output()@out.capture(clear_output=True)def plot_bias(V):    phi, E, info = sim.solve_poisson(float(V))    x = device.grid.x * 1e4        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))        ax1.plot(x, E, 'b-', lw=2)    ax1.set_xlabel('Depth (µm)')    ax1.set_ylabel('E(x) (V/cm)')    ax1.grid(alpha=0.3)    ax1.set_title(f'Electric Field at V = {V:.1f} V')        ax2.plot(x, phi, 'r-', lw=2)    ax2.set_xlabel('Depth (µm)')    ax2.set_ylabel(r'$\phi$(x) (V)')    ax2.grid(alpha=0.3)    ax2.set_title(f'Potential at V = {V:.1f} V')        plt.tight_layout()    plt.show()widget = interactive(plot_bias, V=E_slider)display(VBox([widget.children[0], out]))print("Drag the slider above to explore bias dependence.")

---
## 10. Summary

**The Poisson solver is the engine that drives everything else.**

- It solves the nonlinear 1D Poisson equation with Boltzmann carrier statistics using Newton–Raphson iteration.
- The discretization is a simple finite-volume scheme, giving a **tridiagonal** Jacobian that can be solved with `scipy.linalg.solve_banded`.
- Dirichlet boundary conditions at the contacts come from the doping densities.
- A tanh-step initial guess (or depletion approximation) ensures fast convergence, typically in 2–6 iterations.
- The electric field $E(x) = -d\phi/dx$ is the single input that every subsequent physics module (breakdown, ionization, dark current, trigger probability, PDE) depends on.

**Next experiment (02):** Using $E(x)$ to find the breakdown voltage — how the simulator sweeps bias and detects the >5× current rise.